## Import bibliotek


In [13]:
import requests 
import pandas as pd

jakies produkty potencjalne (pobrane z ceneo): 
- 124893467
- 106545192
- 32918774
- 83177636
- 91869341

### Wysłanie żądania dostępu do zasobu (strony WWW)

szukamy w dokumentacji biblioteki requests

In [2]:
headers = {
    'Cookie' : 'sv3=1.0_177f49d0-5857-11f0-85b9-6d40108099a7', 
    'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36', 
    'Host' : 'www.ceneo.pl'
}

In [3]:
url = 'https://www.ceneo.pl/124893467#tab=reviews'

response = requests.get(url)
print(response.status_code)
# jak kod jest 200 to jest ok

200


U nas na ceneo ten kod html jest ogolnie interaktywny i jest tez w strukturze dom (Document Object Model)
zeby to tak sciagnac potrzebujemy biblioteki beautiful soup


In [4]:
from bs4 import BeautifulSoup

In [5]:
page_dom = BeautifulSoup(response.text, 'html.parser')
# print(page_dom)
# to jest nasza cala strona


teraz szukamy w dokumentacji jak znalezc to co nas interesuje 
potrzebujemy znalezc class = 'js_product-review'

In [6]:
opinions = page_dom.select('div.js_product-review:not(.user-post--highlight)')
print(opinions)
print(len(opinions))


[<div class="user-post user-post__card js_product-review" data-entry-id="17657913">
<div class="user-post__header">
<div class="js_lazy user-post__avatar user-rank__avatar" data-bg="/Content/img/account/avatar/0.svg"></div>
</div>
<div class="user-post__body">
<div class="user-post__content">
<div>
<span class="user-post__author-name">
l...z
</span>
<span class="user-post__score">
<span class="screen-reader-text">Ocena:</span>
<span class="score-container score-container--s js_score-container">
<span class="score-marker score-marker--s" style="width: 100.0%"></span>
</span>
<span class="user-post__score-count">5/5</span>
<span class="user-post__author-recomendation">
<em class="recommended">Polecam</em>
</span>
</span>
</div>
<div class="mb-4">
<span class="verified-purchase">
Zaufana Opinia potwierdzona zakupem
</span>
<span class="user-post__published">
<span class="user-post__published">
Wystawiono
<time datetime="2023-06-28 22:50:27">3 lata temu, </time>
<time datetime="2023-06-12 

In [7]:
opinion = page_dom.select_one('div', class_ = 'js_product-review')
print(type(opinion))

<class 'bs4.element.Tag'>


# Analiza struktury pojedynczej opinii


| składowa | nazwa | selektor |
|----------|-------|----------|
|opinia|opinion|`div.js_product-review`|
| identyfikator | opinion_id | `div.js_product-review` → atrybut `data-entry-id` |
| autor | author | `span.user-post__author-name` |
| treść | content | `div.user-post__text` |
| ocena | score | `span.user-post__score-count` |
| rekomendacja | recommendation | `span.user-post__author-recomendation > em` |
| lista zalet | pros | `div.review-feature__item--positive` |
| lista wad | cons | `div.review-feature__item--negative` |
| dla ilu przydatna | thumbs_up | `button.vote-yes > span` |
| dla ilu nieprzydatna | thumbs_down | `button.vote-no > span` |
| data zamieszczenia | post_date | `span.user-post__published time:nth-of-type(1)` → atrybut `datetime` |
| data zakupu | purchase_date | `span.user-post__published time:nth-of-type(2)` → atrybut `datetime` |


In [27]:
def extract_opinion(opinion_tag):
    
    # Identyfikator — atrybut tagu kontenera
    opinion_id = opinion_tag.get("data-entry-id")

    # Autor — tekst ze spana
    author = opinion_tag.select_one("span.user-post__author-name")
    author = author.get_text(strip=True) if author else None

    # Treść — tekst z diva
    content = opinion_tag.select_one("div.user-post__text")
    content = content.get_text(strip=True) if content else None

    # Ocena — tekst w formacie "5/6", więc wyciągamy samą liczbę
    score = opinion_tag.select_one("span.user-post__score-count")
    score = score.get_text(strip=True).split("/")[0] if score else None

    # Rekomendacja — tekst "Polecam" / "Nie polecam"
    recommendation = opinion_tag.select_one("span.user-post__author-recomendation > em")
    recommendation = recommendation.get_text(strip=True) if recommendation else None

    # Zalety — lista tekstów
    pros = opinion_tag.select("div.feature__item--positives")
    pros = [li.get_text(strip=True) for li in pros] if pros else []

    # Wady — lista tekstów
    cons = opinion_tag.select("div.feature__item--negatives")
    cons = [li.get_text(strip=True) for li in cons] if cons else []

    # Kciuki — liczby całkowite
    thumbs_up = opinion_tag.select_one("button.vote-yes > span")
    thumbs_up = int(thumbs_up.get_text(strip=True)) if thumbs_up else 0

    thumbs_down = opinion_tag.select_one("button.vote-no > span")
    thumbs_down = int(thumbs_down.get_text(strip=True)) if thumbs_down else 0

    # Daty — atrybut datetime tagu <time>, nie tekst widoczny na stronie
    times = opinion_tag.select("span.user-post__published time")
    post_date = times[0].get("datetime") if len(times) > 0 else None
    purchase_date = times[1].get("datetime") if len(times) > 1 else None

    return {
        "opinion_id": opinion_id,
        "author": author,
        "content": content,
        "score": score,
        "recommendation": recommendation,
        "pros": pros,
        "cons": cons,
        "thumbs_up": thumbs_up,
        "thumbs_down": thumbs_down,
        "post_date": post_date,
        "purchase_date": purchase_date,
    }

In [28]:

data = [extract_opinion(op) for op in opinions]
data_df = pd.DataFrame(data)

data_df.head(3)

,opinion_id,author,content,score,recommendation,pros,cons,thumbs_up,thumbs_down,post_date,purchase_date
0,17657913,l...z,#Promocja-HP\nWszystko w jak najlepszym porząd...,5,Polecam,[],[],0,0,2023-06-28 22:50:27,2023-06-12 10:14:50
1,19466347,Tomasz,To moja pierwsza drukarka atramentowa od lat. ...,5,Polecam,[],[],0,0,2025-02-21 10:04:13,2025-02-05 14:16:00
2,17905905,p...k,"Drukarka fajna, super uzupełnianie tuszu, tusz...",3,NaN,[],[],1,0,2023-09-15 11:36:42,2023-09-10 12:31:48
